In [48]:
import pandas as pd
import finances.configs.io as pio

In [ ]:
dfs = pd.read_csv(pio.path_processed_FinanceDatabase_symbols)
symbols = dfs['symbol'].tolist()
dfs.info()

<class 'pandas.DataFrame'>
RangeIndex: 185585 entries, 0 to 185584
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   symbol          185585 non-null  str  
 1   name            168482 non-null  str  
 2   currency        167950 non-null  str  
 3   summary         119731 non-null  str  
 4   category_group  150911 non-null  str  
 5   category        141955 non-null  str  
 6   exchange        180428 non-null  str  
 7   type            185585 non-null  str  
dtypes: str(8)
memory usage: 62.7 MB


In [50]:
# ¡Sin API key! Funciona con "demo"
url = "https://www.alphavantage.co/query?function=LISTING_STATUS&apikey=demo"
df = pd.read_csv(url)
# data preparation
df = df[df['symbol'].isnull() == False]
df = df[df['status'] == 'Active']
df.rename(columns={'assetType': 'type'}, inplace=True)
df.drop(columns=['delistingDate', 'status', "ipoDate"], inplace=True)
df.loc[df["type"] == "ETF", "type"] = "etf"
df.loc[df["type"] == "Stock", "type"] = "stock"
# filter if symbol is in symbols 
df = df[~df.symbol.isin(symbols)]
# info
df.info()

<class 'pandas.DataFrame'>
Index: 11246 entries, 0 to 13263
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   symbol    11246 non-null  str  
 1   name      11183 non-null  str  
 2   exchange  11246 non-null  str  
 3   type      11246 non-null  str  
dtypes: str(4)
memory usage: 940.3 KB


In [51]:
df.isnull().sum() / len(df) 

symbol      0.000000
name        0.005602
exchange    0.000000
type        0.000000
dtype: float64

In [ ]:
assert df.shape[0] == df[df.symbol.isin(symbols)].shape[0] + df[~df.symbol.isin(symbols)].shape[0]
df.shape[0], df[df.symbol.isin(symbols)].shape[0], df[~df.symbol.isin(symbols)].shape[0]

(11246, 0, 11246)

In [54]:
dfs.shape[0], df.shape[0], dfs.shape[0] + df.shape[0]

(185585, 11246, 196831)

In [59]:
dftotal = pd.concat([dfs, df], ignore_index=True)
dftotal.sort_values(by='symbol', inplace=True)
dftotal.reset_index(drop=True, inplace=True)
dftotal.info()  

<class 'pandas.DataFrame'>
RangeIndex: 196831 entries, 0 to 196830
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   symbol          196831 non-null  str  
 1   name            179665 non-null  str  
 2   currency        167950 non-null  str  
 3   summary         119731 non-null  str  
 4   category_group  150911 non-null  str  
 5   category        141955 non-null  str  
 6   exchange        191674 non-null  str  
 7   type            196831 non-null  str  
dtypes: str(8)
memory usage: 64.0 MB


In [63]:
if dftotal["symbol"].duplicated().sum() == 0:
    print("No hay símbolos duplicados.")

No hay símbolos duplicados.
